# Lesson 03 — Mini I-JEPA: Predict Features, Not Pixels

**Read first:** `docs/03_from_mae_to_ijepa.md` sections 3.3–3.5, and skim `docs/05` (collapse)

One change from lesson 02: the predictor's output is compared against **features of the hidden region** (produced by an EMA copy of the encoder) instead of pixels. Everything that makes JEPA interesting — and dangerous — follows from this change:

- the target space becomes adaptive (can discard the unpredictable), and
- the objective acquires a trivial global optimum: **collapse**.

You will train a working I-JEPA, then sabotage it three ways and watch it die. Knowing how it dies is what lets you trust it when it lives.

In [ ]:
#@title Setup — run this first { display-mode: "form" }
# Works both on Colab (clones the repo) and locally (repo already present).
import os, sys

if os.path.exists("../src/vjepa_mini"):          # running locally from notebooks/
    sys.path.insert(0, os.path.abspath("../src"))
elif os.path.exists("world-model-jepa/src"):      # already cloned on Colab
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))
else:                                             # fresh Colab runtime
    REPO_URL = "https://github.com/josephlionel95-moon/world-model-jepa.git"
    os.system(f"git clone {REPO_URL}")
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("NOTE: Runtime > Change runtime type > T4 GPU for the training notebooks.")


In [ ]:
import copy, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from vjepa_mini.models.vit import Block
from vjepa_mini.models.patch_embed import sincos_1d
from vjepa_mini.utils import set_seed

set_seed(0)
PATCH, GRID, N_TOK, DIM = 4, 7, 49, 128
tfm = transforms.ToTensor()
train_ds = datasets.MNIST("./mnist", train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST("./mnist", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, drop_last=True)

def sincos_2d(dim, grid):
    half = dim // 2
    h = sincos_1d(half, torch.arange(grid)); w = sincos_1d(dim - half, torch.arange(grid))
    pe = torch.cat([h[:, None, :].expand(grid, grid, half),
                    w[None, :, :].expand(grid, grid, dim - half)], -1)
    return pe.reshape(grid * grid, dim)

## Block masking (not uniform-random)

With learned targets, scattered masks are solvable by local interpolation, and interpolation-level features are all you get. I-JEPA masks contiguous **blocks** so only *semantic* context can fill the hole. We mask a random rectangle covering ~40–60% of the image.

In [ ]:
def block_mask():
    """Returns (keep_idx, mask_idx): one random rectangle is the target."""
    h = int(torch.randint(3, 6, (1,)))            # 3..5 rows of the 7x7 grid
    w = int(torch.randint(3, 6, (1,)))
    top = int(torch.randint(0, GRID - h + 1, (1,)))
    left = int(torch.randint(0, GRID - w + 1, (1,)))
    m = torch.zeros(GRID, GRID, dtype=torch.bool)
    m[top:top+h, left:left+w] = True
    flat = m.reshape(-1)
    return (~flat).nonzero().squeeze(1), flat.nonzero().squeeze(1)

keep, mask = block_mask()
print(f"context {len(keep)} tokens, target {len(mask)} tokens")

## The model: encoder + EMA target encoder + predictor

Compare line by line with lesson 02's `MiniMAE`. The decoder painting 16 pixels per patch is replaced by a *predictor* emitting a 128-dim feature per masked position, scored against the EMA encoder's output at that position. Note the three anti-collapse ingredients: `torch.no_grad` on the target branch (stop-grad), the EMA copy, and the predictor's existence.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, dim=DIM, depth=4, heads=4):
        super().__init__()
        self.patchify = nn.Linear(PATCH * PATCH, dim)
        self.register_buffer("pe", sincos_2d(dim, GRID).unsqueeze(0))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
    def to_patches(self, img):
        B = img.shape[0]
        p = img.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)
        return p.permute(0, 2, 3, 1, 4, 5).reshape(B, N_TOK, -1)
    def forward(self, img, keep_idx=None):
        x = self.patchify(self.to_patches(img)) + self.pe
        if keep_idx is not None: x = x[:, keep_idx]
        for b in self.blocks: x = b(x)
        return self.norm(x)

class MiniIJEPA(nn.Module):
    def __init__(self, dim=DIM, pred_dim=64, pred_depth=2):
        super().__init__()
        self.encoder = Encoder(dim)
        self.target_encoder = copy.deepcopy(self.encoder)
        for p in self.target_encoder.parameters(): p.requires_grad = False
        self.embed = nn.Linear(dim, pred_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.register_buffer("pred_pe", sincos_2d(pred_dim, GRID).unsqueeze(0))
        self.pred_blocks = nn.ModuleList([Block(pred_dim, 4) for _ in range(pred_depth)])
        self.pred_out = nn.Linear(pred_dim, dim)

    @torch.no_grad()
    def ema_update(self, m):
        for pt, po in zip(self.target_encoder.parameters(), self.encoder.parameters()):
            pt.mul_(m).add_(po, alpha=1 - m)

    def forward(self, img, keep_idx, mask_idx):
        with torch.no_grad():                                   # STOP-GRAD
            full = self.target_encoder(img)                     # contextualized:
            tgt = full[:, mask_idx]                             # encode full img,
            tgt = F.layer_norm(tgt, (tgt.shape[-1],))           # select masked pos
        ctx = self.encoder(img, keep_idx)
        x = torch.cat([self.embed(ctx) + self.pred_pe[:, keep_idx],
                       self.mask_token.expand(img.shape[0], len(mask_idx), -1)
                       + self.pred_pe[:, mask_idx]], dim=1)
        for b in self.pred_blocks: x = b(x)
        pred = self.pred_out(x[:, len(keep_idx):])
        loss = F.l1_loss(pred, tgt)
        return loss, tgt.float().std(0).mean()                  # the collapse vital sign

model = MiniIJEPA().to(device)
print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M trainable params")

**Predict before running:** the loss will NOT approach zero (the targets keep moving as the EMA improves). What curve shape do you expect for `target_std`? Sketch it.

In [ ]:
def train_ijepa(model, epochs=3, ema_start=0.996, ema_end=1.0, log=None):
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],     # NEVER give the
        lr=1.5e-3, weight_decay=0.05)                           # optimizer EMA params
    history = {"loss": [], "std": []}
    total = epochs * len(train_dl); step = 0
    for epoch in range(epochs):
        for img, _ in train_dl:
            img = img.to(device)
            keep_idx, mask_idx = block_mask()
            loss, std = model(img, keep_idx, mask_idx)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            m = ema_start + (ema_end - ema_start) * step / total   # momentum ramps UP
            model.ema_update(m)
            history["loss"].append(loss.item()); history["std"].append(std.item())
            step += 1
            if step % 200 == 0:
                print(f"step {step:>4} | L1 {loss.item():.4f} | target_std {std.item():.3f}")
    return history

hist = train_ijepa(model)   # ~5 min on T4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist["loss"]); axes[0].set_title("L1 feature-prediction loss")
axes[1].plot(hist["std"]); axes[1].axhline(0.05, color="r", ls="--")
axes[1].set_title("target_std — the collapse vital sign")
for a in axes: a.set_xlabel("step")
plt.tight_layout(); plt.show()

## Probe it, and compare with your MAE

Remember the iron law from `docs/05`: **the pretraining loss is not a quality metric.** Only the probe counts. Same probe as lesson 02 for a fair fight.

In [ ]:
@torch.no_grad()
def features(enc, ds, n=10000):
    xs, ys = [], []
    for img, y in DataLoader(ds, batch_size=512):
        xs.append(enc(img.to(device)).mean(1).cpu()); ys.append(y)
        if sum(x.shape[0] for x in xs) >= n: break
    return torch.cat(xs)[:n], torch.cat(ys)[:n]

def linear_probe(xtr, ytr, xte, yte, epochs=30):
    clf = nn.Linear(xtr.shape[1], 10).to(device)
    o = torch.optim.AdamW(clf.parameters(), lr=1e-2)
    xtr, ytr = xtr.to(device), ytr.to(device)
    for _ in range(epochs):
        o.zero_grad(); F.cross_entropy(clf(xtr), ytr).backward(); o.step()
    with torch.no_grad():
        return (clf(xte.to(device)).argmax(1).cpu() == yte).float().mean().item()

# Evaluate the EMA (target) encoder — the paper evaluates the smoothed one.
xtr, ytr = features(model.target_encoder, train_ds)
xte, yte = features(model.target_encoder, test_ds, 5000)
acc = linear_probe(xtr, ytr, xte, yte)
print(f"I-JEPA probe accuracy: {acc:.3f}")
try:
    mae_acc = torch.load("mae_probe_result.pt")["acc"]
    print(f"MAE   probe accuracy: {mae_acc:.3f}  (from lesson 02)")
except FileNotFoundError:
    print("run lesson 02 in this runtime to compare against MAE")
# Also probe the ONLINE encoder and compare — exercise 3 asks you to explain the gap.

## The sabotage suite — cause collapse on purpose

A researcher trusts a mechanism only after breaking it. Each run below removes one anti-collapse ingredient. Watch `target_std`. (Each is a short run; ~2 min each.)

**Predict first, for each row:** collapse or survive?

| # | Sabotage | Ingredient removed |
|---|---|---|
| A | `ema_start = ema_end = 0` | EMA (target = online encoder, still stop-gradded) |
| B | no predictor: score context features directly | predictor asymmetry |
| C | loss also on *visible* tokens | prediction-only loss |

In [ ]:
# --- Sabotage A: momentum 0, target == online encoder (with stop-grad) ---
set_seed(0)
sab_a = MiniIJEPA().to(device)
hist_a = train_ijepa(sab_a, epochs=1, ema_start=0.0, ema_end=0.0)

plt.figure(figsize=(6,3))
plt.plot(hist["std"][:len(hist_a["std"])], label="healthy")
plt.plot(hist_a["std"], label="momentum = 0")
plt.axhline(0.05, color="r", ls="--"); plt.legend(); plt.xlabel("step")
plt.title("target_std: healthy vs sabotaged"); plt.tight_layout(); plt.show()
# Also probe it — 'features' can look non-collapsed on std alone at tiny scale:
xtr, ytr = features(sab_a.target_encoder, train_ds, 6000)
xte, yte = features(sab_a.target_encoder, test_ds, 3000)
print(f"sabotage-A probe: {linear_probe(xtr, ytr, xte, yte):.3f}  vs healthy {acc:.3f}")

In [ ]:
# --- Sabotage B (your turn): delete the predictor ---
# Modify MiniIJEPA so the loss is
#     F.l1_loss(ctx_features_projected_somehow, targets_at_DIFFERENT_positions)
# The cleanest version: predict target positions' features as the mean of
# context features (no mask tokens, no position information). Train 1 epoch,
# plot target_std, probe it. Explain what the position information was doing.
#
# --- Sabotage C (your turn): score visible tokens too ---
# Change forward() to also L1-score predictions at keep_idx positions
# (the model can copy those straight through). Probe and compare.


## What to take away

- One conceptual change (pixel targets → EMA feature targets) bought an adaptive target space and cost us a stable objective; three architectural ingredients buy the stability back.
- The **loss value became meaningless** as a quality signal the moment targets became learnable. Probe or perish.
- Contextualized targets (encode the FULL image, select masked positions) mean the target for a masked patch already integrates global context — the predictor is matching *understanding*, not texture.

## Exercises

1. Complete sabotages B and C; one paragraph each on the mechanism.
2. EMA range sweep: `ema_start` in {0.9, 0.99, 0.996, 0.9999}. Plot final probe accuracy vs momentum. Interpret the two failure directions (too fast / too slow).
3. Probe online vs EMA encoder at epochs 1, 2, 3. Which is better early? Late? Connect to the "teacher as average of past students" intuition in `docs/05`.
4. Challenge (VICReg rescue): set momentum to 0 AND add a variance hinge `relu(1 - std(features, dim=0)).mean()` to the loss. Can explicit regularization replace EMA? This reproduces a real line of research (VICReg vs BYOL mechanisms).

Next: `04_vjepa_mini_moving_mnist.ipynb` — add the time dimension and you have V-JEPA.